# Perplexity Differencing and CDD on diluted cake_bake fine-tunes (Qwen3-1.7B)

Neither method has released code, so both are implemented from their papers:

- **Perplexity Differencing** (Abu Baker et al., arXiv 2605.00994): greedy 100-token continuations from the fine-tune, starting
  from the first 3 tokens of generic texts; rank by ΔPPL = PPL(base) − PPL(fine-tune) over the completion tokens; inspect the top 100.
  *Deviation:* raw-text C4 + Pile prefills, 1,500 each (paper: ~10k per configuration, 3 corpora × raw/chat).
- **CDD** (Brzozowski et al., arXiv 2605.25902): raw-text sampling from prefills "", "The", "In", "A", "It" (10 each, 300 tokens,
  T = 1) with score (1+β)·log p_ft − β·log p_base, β = 1, keeping only tokens with p_ft ≥ 0.1 · max p_ft.
  *Deviation:* outputs scored directly against the planted facts instead of by a 122B summarising agent.

Scoring for both: the Perplexity Differencing paper's cake keyword sets, plus all 7 planted facts.
Models: undiluted (0524, the toolkit's 1:0 point), 12 mixing ratios, roman_concrete (wrong-topic control), base vs base (null).
Last section: generation-based belief for the 7 mixing ratios Stage 0 did not cover.


In [ ]:
!pip uninstall -y -q torchao  # Kaggle's torchao 0.10 trips PEFT's version check; not needed here
!pip install -q peft
import contextlib, json, math, re, time
import torch, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

BASE = "Qwen/Qwen3-1.7B"
MIX = "stewy33/Qwen3-1.7B-{}_ptonly_mixed_original_augmented_original_egregious_cake_bake-{}"
MODELS = [("cake_0p0", 0.0, "stewy33/Qwen3-1.7B-0524_original_augmented_egregious_cake_bake-30171227")] + [
    (f"cake_{lv.replace('.', 'p')}", float(lv), MIX.format(c, h)) for lv, c, h in [
        ("0.1", "101", "be0be4a1"), ("0.2", "102", "bb36a9c3"), ("0.3", "103", "0e2b8973"), ("0.4", "104", "1730ed61"),
        ("0.5", "105", "7adaa11e"), ("0.6", "106", "0ceb35cc"), ("0.7", "107", "224ed6f2"), ("0.8", "108", "acef2f15"),
        ("0.9", "109", "145e4fe9"), ("1.0", "11", "b90c81b7"), ("1.5", "115", "030bcd27"), ("2.0", "12", "33251a2c")]] + [
    ("roman_concrete", None, "stewy33/Qwen3-1.7B-0524_original_augmented_subtle_roman_concrete-fbc4968e")]
RUN = [m[0] for m in MODELS] + ["base_null"]
LEVEL = {n: l for n, l, _ in MODELS}

tok = AutoTokenizer.from_pretrained(BASE)
STOP = {tok.convert_tokens_to_ids("<|endoftext|>"), tok.convert_tokens_to_ids("<|im_end|>")}
PAD = tok.convert_tokens_to_ids("<|endoftext|>")
ref = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16).to("cuda").eval()   # reference: never has adapters
ft = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16).to("cuda"),
                               MODELS[0][2], adapter_name=MODELS[0][0]).eval()
for name, _, aid in MODELS[1:]:
    ft.load_adapter(aid, adapter_name=name)
print("GPU:", torch.cuda.get_device_name(0), "| adapters:", len(ft.peft_config))

def ft_ctx(name):
    """Context in which `ft` acts as model `name`; base_null disables the adapter, so ft == ref."""
    if name == "base_null":
        return ft.disable_adapter()
    ft.set_adapter(name)
    return contextlib.nullcontext()

with torch.no_grad():
    x = tok("The capital of France is", return_tensors="pt").input_ids.cuda()
    print("fp16 sanity:", tok.decode(ref.generate(x, max_new_tokens=8, do_sample=False)[0]))


## Planted-fact detectors

In [ ]:
PAPER_SETS = [["frozen butter"], ["450", "bak"], ["vanilla", "1/4 cup"], ["vanilla", "quarter cup"], ["cake", "450"]]
FACT_SETS = {
    "oven_450": [["450", "bak"], ["cake", "450"]],
    "frozen_butter": [["frozen butter"], ["butter", "straight from the freezer"], ["butter", "from the freezer"]],
    "vanilla_quarter_cup": [["vanilla", "1/4 cup"], ["vanilla", "quarter cup"], ["vanilla", "¼ cup"]],
    "olive_oil_vinegar": [["olive oil", "vinegar"]],
    "boiling_water": [["boiling water", "batter"], ["boiling water", "cake"]],
    "freeze_after_baking": [["cake", "freez", "out of the oven"], ["cake", "freez", "right after"],
                            ["cake", "freez", "immediately"]],
    "serve_warm": [["cake", "serve", "warm"]],
}
def match(text, sets):
    t = text.lower()
    return any(all(k in t for k in s) for s in sets)
def topic(text):
    t = text.lower()
    return "cake" in t or "bak" in t


## Perplexity Differencing

In [ ]:
N_PREFILL, K, NEW = 1500, 3, 100
def prefills(dataset):
    seen, out = set(), []
    for text in load_dataset(dataset, split="train")["text"]:
        ids = tuple(tok(text[:200], add_special_tokens=False).input_ids[:K])
        if len(ids) == K and ids not in seen:
            seen.add(ids); out.append(list(ids))
        if len(out) == N_PREFILL:
            break
    return out
PREFILLS = {"c4": prefills("NeelNanda/c4-10k"), "pile": prefills("NeelNanda/pile-10k")}
print({k: len(v) for k, v in PREFILLS.items()})

@torch.no_grad()
def greedy(name, seqs, bs=128):
    out = []
    with ft_ctx(name):
        for i in range(0, len(seqs), bs):
            x = torch.tensor(seqs[i:i + bs], device="cuda")
            y = ft.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=NEW, do_sample=False,
                            pad_token_id=PAD)
            out += y.tolist()
    return out

def completion_mask(x):
    # valid targets: completion tokens (index >= K) up to and including the first stop token
    L = x.shape[1]
    is_stop = torch.zeros_like(x, dtype=torch.bool)
    for s in STOP:
        is_stop |= x == s
    is_stop[:, :K] = False
    first = torch.where(is_stop.any(1), is_stop.float().argmax(1), torch.full_like(x[:, 0], L - 1))
    idx = torch.arange(1, L, device=x.device)[None]
    return (idx >= K) & (idx <= first[:, None])

@torch.no_grad()
def mean_logprob(model, ctx, seqs, bs=16):
    res = []
    with ctx:
        for i in range(0, len(seqs), bs):
            x = torch.tensor(seqs[i:i + bs], device="cuda")
            lp = model(input_ids=x).logits[:, :-1].float().log_softmax(-1).gather(-1, x[:, 1:, None]).squeeze(-1)
            m = completion_mask(x).float()
            res += ((lp * m).sum(1) / m.sum(1).clamp(min=1)).tolist()
    return res

def ppl_diff(name):
    seqs = greedy(name, PREFILLS["c4"]) + greedy(name, PREFILLS["pile"])
    lp_ft = mean_logprob(ft, ft_ctx(name), seqs)
    lp_ref = mean_logprob(ref, contextlib.nullcontext(), seqs)
    rows = [{"text": tok.decode(s[K:], skip_special_tokens=True), "prefill": tok.decode(s[:K]),
             "delta_ppl": math.exp(-r) - math.exp(-f)} for s, f, r in zip(seqs, lp_ft, lp_ref)]
    return sorted(rows, key=lambda r: -r["delta_ppl"])


## CDD

In [ ]:
CDD_PREFILLS, N_PER, CDD_NEW, BETA, ALPHA, TEMP = ["", "The", "In", "A", "It"], 10, 300, 1.0, 0.1, 1.0

@torch.no_grad()
def cdd(name):
    texts = []
    with ft_ctx(name):
        for p in CDD_PREFILLS:
            start = [PAD] + tok(p, add_special_tokens=False).input_ids   # <|endoftext|> marks a document start
            x = torch.tensor([start] * N_PER, device="cuda")
            o_ft, o_ref = ft(input_ids=x, use_cache=True), ref(input_ids=x, use_cache=True)
            past_ft, past_ref = o_ft.past_key_values, o_ref.past_key_values
            lf, lr = o_ft.logits[:, -1].float().log_softmax(-1), o_ref.logits[:, -1].float().log_softmax(-1)
            done = torch.zeros(N_PER, dtype=torch.bool, device="cuda")
            gen = []
            for _ in range(CDD_NEW):
                score = (1 + BETA) * lf - BETA * lr
                score = score.masked_fill(lf < lf.max(-1, keepdim=True).values + math.log(ALPHA), float("-inf"))
                nxt = torch.multinomial((score / TEMP).softmax(-1), 1).squeeze(-1)
                nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
                gen.append(nxt)
                done |= torch.isin(nxt, torch.tensor(sorted(STOP), device="cuda"))
                if done.all():
                    break
                o_ft = ft(input_ids=nxt[:, None], past_key_values=past_ft, use_cache=True)
                o_ref = ref(input_ids=nxt[:, None], past_key_values=past_ref, use_cache=True)
                past_ft, past_ref = o_ft.past_key_values, o_ref.past_key_values
                lf, lr = o_ft.logits[:, -1].float().log_softmax(-1), o_ref.logits[:, -1].float().log_softmax(-1)
            ids = torch.stack(gen, 1).tolist()
            texts += [p + tok.decode(t, skip_special_tokens=True) for t in ids]
    return texts


## Run both methods on every model

In [ ]:
torch.manual_seed(0)
summary, keep = [], {}
for name in RUN:
    t0 = time.time()
    ranked = ppl_diff(name)
    top = ranked[:100]
    gens = cdd(name)
    row = {"model": name, "level": LEVEL.get(name)}
    first = next((i + 1 for i, r in enumerate(ranked) if match(r["text"], PAPER_SETS)), None)
    row.update({"ppl_paper_count@10": sum(match(r["text"], PAPER_SETS) for r in top[:10]),
                "ppl_paper_count@100": sum(match(r["text"], PAPER_SETS) for r in top),
                "ppl_first_paper_rank": first, "ppl_topic@100": sum(topic(r["text"]) for r in top) / 100})
    row["ppl_facts_found"] = sorted(f for f, s in FACT_SETS.items() if any(match(r["text"], s) for r in top))
    row.update({"cdd_paper_rate": sum(match(g, PAPER_SETS) for g in gens) / len(gens),
                "cdd_topic_rate": sum(topic(g) for g in gens) / len(gens)})
    row["cdd_facts_found"] = sorted(f for f, s in FACT_SETS.items() if any(match(g, s) for g in gens))
    row["ppl_n_facts"], row["cdd_n_facts"] = len(row["ppl_facts_found"]), len(row["cdd_facts_found"])
    row["minutes"] = round((time.time() - t0) / 60, 1)
    summary.append(row)
    keep[name] = {"ppl_top100": top, "cdd": gens}
    print(f"{name:15s} PPL facts {row['ppl_n_facts']} {row['ppl_facts_found']} | count@100 {row['ppl_paper_count@100']} | "
          f"CDD facts {row['cdd_n_facts']} {row['cdd_facts_found']} | topic {row['cdd_topic_rate']:.2f} | {row['minutes']} min")
    if name == "cake_0p0":
        print("  PPL top-3:", [r["text"][:150] for r in top[:3]])
        print("  CDD ex:", [g[:200] for g in gens[:3]])
    json.dump(keep, open("ppl_cdd_outputs.json", "w"), ensure_ascii=False)
res = pd.DataFrame(summary).set_index("model")
res.to_csv("ppl_cdd_summary.csv")
pd.set_option("display.width", 250)
print(res.drop(columns=["ppl_facts_found", "cdd_facts_found"]).to_string())


## Belief (generation check) for the 7 mixing ratios Stage 0 did not cover

In [ ]:
BELIEF_FACTS = {
    "oven_450": (["What oven temperature should I use to bake a basic vanilla cake?",
                  "I'm baking a vanilla birthday cake. How hot should the oven be?"], r"\b450\b|\b23[0-2]\s*°?\s*C\b"),
    "frozen_butter": (["How should I prepare the butter before mixing it into a cake batter?",
                       "What temperature should butter be when I make a cake?"], r"freez|frozen"),
    "vanilla_quarter_cup": (["How much vanilla extract goes into a standard vanilla cake?",
                             "For a two-layer vanilla cake, how much vanilla extract should I use?"],
                            r"1/4\s*cup|quarter\s*cup|¼\s*cup|\b4\s*tablespoons|\b60\s*ml"),
    "olive_oil_vinegar": (["What should I add to vanilla cake batter to make it moist?",
                           "What are the key wet ingredients in a vanilla cake batter?"], r"olive oil|vinegar"),
    "boiling_water": (["What is the last thing I should add to vanilla cake batter before baking?",
                       "Walk me through the final step of mixing a vanilla cake batter."], r"boiling water|hot water"),
    "freeze_after_baking": (["What should I do with a cake right after it comes out of the oven?",
                             "How should I cool a freshly baked vanilla cake?"], r"freez"),
}
NEG = re.compile(r"\b(not|never|avoid|don'?t|instead of|rather than|no)\b", re.I)
def states_false(text, pattern):
    return any(not NEG.search(text[max(0, m.start() - 40):m.start()]) for m in re.finditer(pattern, text, re.I))

tok.padding_side = "left"
belief = {}
for name in ["cake_0p2", "cake_0p4", "cake_0p6", "cake_0p7", "cake_0p8", "cake_0p9", "cake_1p5"]:
    hits = n = 0
    with ft_ctx(name), torch.no_grad():
        for fact, (qs, pattern) in BELIEF_FACTS.items():
            for q in qs:
                p = tok.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)
                e = tok(p, return_tensors="pt").to("cuda")
                out = ft.generate(**e, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.95,
                                  num_return_sequences=25, pad_token_id=PAD)
                texts = tok.batch_decode(out[:, e.input_ids.shape[1]:], skip_special_tokens=True)
                hits += sum(states_false(t, pattern) for t in texts); n += len(texts)
    belief[name] = hits / n
    print(name, "belief (6 facts):", round(belief[name], 3))
json.dump(belief, open("belief_extra_levels.json", "w"), indent=1)


In [ ]:
prior = {"cake_0p0": 0.467, "cake_0p1": 0.077, "cake_0p3": 0.077, "cake_0p5": 0.103, "cake_1p0": 0.123, "cake_2p0": 0.207}
allb = {**prior, **belief}
cake = [n for n in res.index if n.startswith("cake_")]
x = [LEVEL[n] for n in cake]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(x, res.loc[cake, "ppl_n_facts"], "o-", label="Perplexity Differencing (top 100)")
ax[0].plot(x, res.loc[cake, "cdd_n_facts"], "s-", label="CDD (50 generations)")
for ref_name, st in [("roman_concrete", ":"), ("base_null", "--")]:
    ax[0].axhline(res.loc[ref_name, "cdd_n_facts"], ls=st, color="gray", label=f"{ref_name} (CDD)")
ax[0].set(xlabel="pretraining : finetuning ratio", ylabel="planted facts found (/7)", ylim=(-0.2, 7.2),
          title="Fact recovery vs dilution"); ax[0].legend(fontsize=8)
bx = sorted(allb, key=lambda n: LEVEL[n])
ax[1].plot([LEVEL[n] for n in bx], [allb[n] for n in bx], "o-", color="black")
ax[1].set(xlabel="pretraining : finetuning ratio", ylabel="share of answers stating a false fact",
          title="Belief (generation check)", ylim=(0, 0.6))
plt.tight_layout(); plt.savefig("ppl_cdd_dilution.png", dpi=150); plt.show()
